In [60]:
# imports
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import geopandas as gpd

In [61]:
df = pd.read_csv("/Users/khuenguyen/Desktop/team-nuclear-family/processed_data/data_centers.csv")
df.head()

,id,type,lat,lon,name,operator,city,state_abbr
0,751881301,node,38.902888,-77.029149,CoreSite DC1,CoreSite,NaN,NaN
1,1825907839,node,35.103293,-106.589349,Southwest Cyberport,NaN,Albuquerque,NM
2,2590601520,node,42.437882,-123.328630,FCR,NaN,Grants Pass,OR
3,2607827436,node,32.775025,-117.070888,SDSU Computer Room,NaN,NaN,NaN
4,3406467087,node,25.791545,-80.317748,NocRoom Miami IT Services,NaN,NaN,NaN


In [62]:
gdf = gpd.GeoDataFrame(
    df,
    geometry = gpd.points_from_xy(df["lon"], df["lat"]),
    crs = "EPSG:4326"
)

gdf = gdf[["id", "name", "geometry"]]
gdf

,id,name,geometry
0,751881301,CoreSite DC1,POINT (-77.02915 38.90289)
1,1825907839,Southwest Cyberport,POINT (-106.58935 35.10329)
2,2590601520,FCR,POINT (-123.32863 42.43788)
3,2607827436,SDSU Computer Room,POINT (-117.07089 32.77503)
4,3406467087,NocRoom Miami IT Services,POINT (-80.31775 25.79155)
...,...,...,...
1407,1473440676,NaN,POINT (-112.01946 40.26986)
1408,1481581947,CyrusOne CHI3,POINT (-88.24675 41.80286)
1409,1481581948,Edged Chicago ORD01,POINT (-88.2411 41.80693)
1410,1482073995,Verizon,POINT (-78.72879 40.87825)


In [63]:
df_county = gpd.read_file("/Users/khuenguyen/Desktop/team-nuclear-family/raw_data/county_boundaries_2025/tl_2025_us_county.shp")
df_county = df_county[["GEOID", "NAMELSAD", "geometry"]]
df_county

,GEOID,NAMELSAD,geometry
0,40075,Kiowa County,"POLYGON ((-98.95506 35.11643, -98.94903 35.116..."
1,46079,Lake County,"POLYGON ((-96.88886 43.9353, -96.88886 43.9351..."
2,37033,Caswell County,"POLYGON ((-79.14343 36.4422, -79.14345 36.4418..."
3,48377,Presidio County,"POLYGON ((-104.98078 30.62552, -104.98073 30.6..."
4,39057,Greene County,"POLYGON ((-84.10668 39.68891, -84.10662 39.689..."
...,...,...,...
3230,53065,Stevens County,"POLYGON ((-117.86678 47.84506, -117.8669 47.84..."
3231,19177,Van Buren County,"POLYGON ((-92.17907 40.89972, -92.17704 40.899..."
3232,31073,Gosper County,"POLYGON ((-99.98139 40.62501, -99.9814 40.6269..."
3233,28095,Monroe County,"POLYGON ((-88.22909 33.89194, -88.2291 33.8918..."


In [64]:
gdf = gdf.sjoin(df_county.to_crs(gdf.crs), how = "left", predicate = "within")
gdf

,id,name,geometry,index_right,GEOID,NAMELSAD
0,751881301,CoreSite DC1,POINT (-77.02915 38.90289),2395,11001,District of Columbia
1,1825907839,Southwest Cyberport,POINT (-106.58935 35.10329),3212,35001,Bernalillo County
2,2590601520,FCR,POINT (-123.32863 42.43788),1526,41033,Josephine County
3,2607827436,SDSU Computer Room,POINT (-117.07089 32.77503),1896,06073,San Diego County
4,3406467087,NocRoom Miami IT Services,POINT (-80.31775 25.79155),939,12086,Miami-Dade County
...,...,...,...,...,...,...
1407,1473440676,NaN,POINT (-112.01946 40.26986),2453,49049,Utah County
1408,1481581947,CyrusOne CHI3,POINT (-88.24675 41.80286),87,17043,DuPage County
1409,1481581948,Edged Chicago ORD01,POINT (-88.2411 41.80693),87,17043,DuPage County
1410,1482073995,Verizon,POINT (-78.72879 40.87825),2501,42033,Clearfield County


In [65]:
df_data_centers = gdf.groupby(["NAMELSAD", "GEOID"]).size().reset_index()
df_data_centers.columns = ["county_name", "geoid", "data_centers_count"]
df_data_centers

,county_name,geoid,data_centers_count
0,Ada County,16001,2
1,Adams County,08001,1
2,Aiken County,45003,1
3,Alachua County,12001,2
4,Alameda County,06001,5
...,...,...,...
239,White County,05145,1
240,Wichita County,48485,1
241,Williamson County,47187,5
242,Yates County,36123,1


In [66]:
df_data_centers.to_csv("../processed_data/data_centers_by_county.csv")